# Domain B: Anomaly Detection - Exploration

This notebook explores the 10 approaches for anomaly detection.

## Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.domain_b_anomaly_detection.data_generator import create_anomaly_dataset
from src.domain_b_anomaly_detection.approach_01_statistical import StatisticalAnomalyDetector
from src.domain_b_anomaly_detection.approach_03_tree_based import IsolationForestDetector
from src.domain_b_anomaly_detection.approach_04_autoencoder import AutoencoderDetector
from src.core.metrics import compute_anomaly_metrics

## Generate Dataset

In [ ]:
dataset = create_anomaly_dataset(n_train=5000, n_val=1000, n_test=1000)

print(f"Train shape: {dataset['train']['X'].shape}")
print(f"Train anomaly ratio: {dataset['train']['y'].mean():.4f}")
print(f"Test shape: {dataset['test']['X'].shape}")
print(f"Test anomaly ratio: {dataset['test']['y'].mean():.4f}")

## Visualize Data

In [ ]:
X_train = dataset['train']['X']
y_train = dataset['train']['y']

# Plot first two features
plt.figure(figsize=(10, 6))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], 
            c='blue', alpha=0.5, label='Normal')
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], 
            c='red', alpha=0.8, label='Anomaly')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Anomaly Detection Dataset')
plt.legend()
plt.show()

## Compare Approaches

In [ ]:
results = []

# Statistical
stat_model = StatisticalAnomalyDetector({'method': 'zscore'})
stat_model.train(dataset['train']['X'], dataset['train']['y'])
stat_pred = stat_model.predict(dataset['test']['X'])
stat_scores = stat_model.score(dataset['test']['X'])
stat_metrics = compute_anomaly_metrics(dataset['test']['y'], stat_pred, stat_scores)
results.append({'Approach': 'Statistical (Z-score)', **stat_metrics})

# Isolation Forest
iso_model = IsolationForestDetector({'n_estimators': 100})
iso_model.train(dataset['train']['X'], dataset['train']['y'])
iso_pred = iso_model.predict(dataset['test']['X'])
iso_scores = iso_model.score(dataset['test']['X'])
iso_metrics = compute_anomaly_metrics(dataset['test']['y'], iso_pred, iso_scores)
results.append({'Approach': 'Isolation Forest', **iso_metrics})

# Autoencoder
ae_model = AutoencoderDetector({'epochs': 30})
ae_model.train(dataset['train']['X'], dataset['train']['y'])
ae_pred = ae_model.predict(dataset['test']['X'])
ae_scores = ae_model.score(dataset['test']['X'])
ae_metrics = compute_anomaly_metrics(dataset['test']['y'], ae_pred, ae_scores)
results.append({'Approach': 'Autoencoder', **ae_metrics})

# Display results
df = pd.DataFrame(results)
df[['Approach', 'precision', 'recall', 'f1', 'roc_auc']]

## Analyze Trade-offs

In [ ]:
# Philosophy comparison
for model in [stat_model, iso_model, ae_model]:
    philosophy = model.get_philosophy()
    print(f"\n{model.name}:")
    print(f"  Mental Model: {philosophy['mental_model']}")
    print(f"  Strengths: {philosophy['strengths']}")
    print(f"  Weaknesses: {philosophy['weaknesses']}")